In [1]:
!pip install -r requirements.txt

In [3]:
from orchestrator_agent import MultiAgent
import os
from pathlib import Path
from test_cases import TEST_CASES
import json
os.environ["OPENAI_API_KEY"] = Path("open_ai_api_key.txt").read_text().strip()

In [4]:
agent = MultiAgent()


In [6]:
result_traces = []
for case in TEST_CASES: 
    answer,_,trace_info = await agent.answer(case["prompt"])
    # Add the response to the traces
    trace_info["answer"] = answer
    #Append to result traces
    result_traces.append(trace_info)
    print(f"Case {case["id"]} Done")



Case 1 Done
Case 2 Done
Case 3 Done
Case 4 Done
Case 5 Done
Case 6 Done
Case 7 Done
Case 8 Done
Case 9 Done
Case 10 Done
Case 11 Done
Case 12 Done
Case 13 Done
Case 14 Done
Case 15 Done
Case 16 Done
Case 17 Done
Case 18 Done


NameError: name 'json' is not defined

In [7]:

#dump to file 
json.dump(result_traces, open("eval_suite.json", "w"), indent=2, default=str)
#print(result_traces)

[{'question': 'Tell me about the public meeting on beavers', 'tool_calls': [{'name': 'rag', 'input': {'input': 'public meeting on beavers'}, 'output': "I couldn't find any public meeting on beavers in the City of Boston public notices."}, {'name': 'web_search', 'input': {'input': 'public meeting on beavers Boston'}, 'output': 'I couldn’t find any public meeting notices about beavers on the City of Boston public notices page. ([search.boston.gov](https://search.boston.gov/public-notices?utm_source=openai))\n\nThere are 311 reports referencing beaver incidents (e.g., trees or a dead beaver), so the city has received related service requests. ([311.boston.gov](https://311.boston.gov/tickets/101005797316?utm_source=openai))\n\nWould you like me to (a) monitor Boston.gov and notify you if a beaver-related public notice appears, or (b) search broader web sources (outside boston.gov) for meetings or community discussions?'}], 'tools_called': ['rag', 'web_search'], 'sources': [], 'latency_ms':

In [20]:
import pandas as pd

with open("eval_suite.json","r", encoding="utf-8") as file:
    data = json.load(file)

for i in range(len(data)):
    # Stamp the category of the test case
    data[i]["category"] = TEST_CASES[i]["name"]
    # Stamp the response notes from the test case
    data[i]["expected_response_notes"] = TEST_CASES[i]["expectations"]["response_notes"]
    
    # Check if sources expectation matches
    data[i]["source_pass"] = (not (data[i]["sources"]) == (not TEST_CASES[i]["expectations"]["sources"]) )

    # Check guardrails
    guard = data[i].get("guardrail_tripped", False)
    data[i]["guardrail_pass"] = (guard == TEST_CASES[i]["expectations"]["guardrail_trip"])

    # Check expected tools 
    data[i]["tool_pass"]= False
    if TEST_CASES[i]["expectations"]["expected_tools"]:
        for entry in TEST_CASES[i]["expectations"]["expected_tools"]:
            if entry in data[i]["tools_called"]:
                data[i]["tool_pass"]=True
    else:
        data[i]["tool_pass"]=(TEST_CASES[i]["expectations"]["expected_tools"]==data[i]["tools_called"])
 
    
    # Check sources if relevant
    if TEST_CASES[i]["expectations"]["notice_ids"]:
        #Default is fail
        data[i]["correct_notices"] = False
        for entry in data[i]["sources"]:
            if entry["notice_id"] in TEST_CASES[i]["expectations"]["notice_ids"]:
                data[i]["correct_notices"] = True
    else:
        data[i]["correct_notices"] = "N/A"
        

df = pd.DataFrame(data)   

average_latency = df["latency_ms"].mean()
print(f"Average Latency: {average_latency} ms")
    

Average Latency: 15069.238888888889 ms


In [21]:
df

,question,tool_calls,tools_called,sources,latency_ms,answer,category,expected_response_notes,source_pass,guardrail_pass,tool_pass,correct_notices,guardrail_tripped,guardrail_reason
0,Tell me about the public meeting on beavers,"[{'name': 'rag', 'input': {'input': 'public me...","[rag, web_search]",[],30304.2,There doesn't appear to be any specific public...,Non-existant meeting,Should not find any matching public notices,True,True,True,N/A,NaN,NaN
1,Tell me about the city meeting on unicorns sig...,[],[],[],1227.9,"Sorry, that doesn't seem to be related to the ...",Non-existant meeting,Should not find any matching public notices,True,False,False,N/A,True,"The query mentions a city meeting, but the sub..."
2,Can I testify at the August 6th Tree Removal H...,"[{'name': 'rag', 'input': {'input': 'August 6t...",[rag],"[{'notice_id': '16602326', 'title': 'Tree Remo...",9729.6,You cannot testify at the Tree Removal Hearing...,Public testimony,Valid meeting but no public testimony at this ...,True,True,True,True,NaN,NaN
3,Can I testify at the August 11th Zoning Board ...,"[{'name': 'rag', 'input': {'input': 'August 11...",[rag],"[{'notice_id': '16602821', 'title': 'Zoning Bo...",9314.5,"Yes, you can testify at the Zoning Board of Ap...",Public testimony,Valid meeting and public testimony allowed at ...,True,True,True,True,NaN,NaN
4,Is the Boston Landmarks Commission meeting hap...,"[{'name': 'rag', 'input': {'input': 'Boston La...","[rag, web_search]",[],41472.9,The Boston Landmarks Commission does not have ...,Cancelled meeting,Valid meeting but was cancelled,True,True,True,False,NaN,NaN
5,Is the August 19th St Botolph area meeting hap...,"[{'name': 'rag', 'input': {'input': 'August 19...",[rag],"[{'notice_id': '16603531', 'title': 'St.Botolp...",10752.6,The public meeting for the St. Botolph Area Ar...,Cancelled meeting,Valid meeting but was cancelled,True,True,True,True,NaN,NaN
6,What is Docket #0218 from the City Council Com...,"[{'name': 'rag', 'input': {'input': 'Docket #0...",[rag],"[{'notice_id': '16600016', 'title': 'City Coun...",9092.4,Docket #0218 will be discussed at a hearing by...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16600016,True,True,True,True,NaN,NaN
7,What is Docket #0932 from the City Council Com...,"[{'name': 'rag', 'input': {'input': 'Docket #0...",[rag],"[{'notice_id': '16594651', 'title': 'City Coun...",9851.5,Docket #0932 pertains to a hearing set by the ...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16600081,True,True,True,True,NaN,NaN
8,Explain crypto wallets,[],[],[],921.5,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,True,N/A,True,The query about crypto wallets is focused on c...
9,Write a fun limeric,[],[],[],1023.8,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,True,N/A,True,"The query asks for a limerick, which falls und..."
